## Learning Objectives

By the end of this lab, you will be able to:

1. Implement scaled dot-product attention from scratch.
2. Build a reusable multi-head attention module.
3. Visualize attention weights as a heatmap.
4. Train a small attention-based classifier on synthetic text-style data.

## Recommended Notebook Pattern

Structure the notebook as:

1. Setup
2. Core attention implementation
3. Validation on a small example
4. Visualization
5. Model training
6. Interpretation

## Lab Validation Standard

At each stage, include a quick correctness check such as:

- tensor shapes
- attention weight sums
- heatmap output
- training metrics
- one short note on what the attention pattern suggests

## Setup


In [ ]:
#| eval: false
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## Part 1: Scaled Dot-Product Attention


In [ ]:
#| eval: false
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (B, H, Tq, d)
    K: (B, H, Tk, d)
    V: (B, H, Tk, d)
    mask: optional boolean tensor broadcastable to scores shape
    """
    d = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

B, H, T, d = 2, 2, 6, 8
Q = torch.randn(B, H, T, d)
K = torch.randn(B, H, T, d)
V = torch.randn(B, H, T, d)
out, w = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)
print("Weights shape:", w.shape)


## Part 2: Multi-Head Attention Module


In [ ]:
#| eval: false
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.attn_weights = None

    def split_heads(self, x):
        B, T, E = x.shape
        x = x.view(B, T, self.num_heads, self.head_dim)
        return x.permute(0, 2, 1, 3)

    def merge_heads(self, x):
        B, H, T, D = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.view(B, T, H * D)

    def forward(self, x, mask=None):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        out, self.attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        out = self.merge_heads(out)
        return self.W_o(self.dropout(out))

mha = MultiHeadAttention(embed_dim=32, num_heads=4)
x = torch.randn(2, 10, 32)
y = mha(x)
print("MHA output:", y.shape)


## Part 3: Visualize Attention Weights


In [ ]:
#| eval: false
weights = mha.attn_weights[0, 0].detach().cpu()  # first batch, first head

plt.figure(figsize=(6, 5))
plt.imshow(weights, cmap="viridis", aspect="auto")
plt.colorbar(label="Attention weight")
plt.title("Attention Heatmap (Head 0)")
plt.xlabel("Key positions")
plt.ylabel("Query positions")
plt.tight_layout()
plt.show()


## Part 4: Attention-Based Classifier

We create a toy dataset where token 1 implies class 0 and token 2 implies class 1.


In [ ]:
#| eval: false
def make_dataset(n=1500, seq_len=20, vocab_size=100):
    xs, ys = [], []
    for _ in range(n):
        label = torch.randint(0, 2, (1,)).item()
        seq = torch.randint(3, vocab_size, (seq_len,))
        seq[0] = label + 1
        xs.append(seq)
        ys.append(label)
    X = torch.stack(xs)
    y = torch.tensor(ys)
    return TensorDataset(X, y)

class AttentionClassifier(nn.Module):
    def __init__(self, vocab_size=100, embed_dim=32, num_heads=4, num_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.mha = MultiHeadAttention(embed_dim, num_heads)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        h = self.embed(x)
        h = self.norm(h + self.mha(h))
        cls = h[:, 0, :]
        return self.head(cls)

dataset = make_dataset()
train_ds, val_ds = torch.utils.data.random_split(dataset, [1200, 300])
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=64)

model = AttentionClassifier().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(1, 6):
    model.train()
    correct = total = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()
        correct += (logits.argmax(-1) == yb).sum().item()
        total += len(yb)
    train_acc = correct / total

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(-1)
            correct += (pred == yb).sum().item()
            total += len(yb)
    val_acc = correct / total
    print(f"Epoch {epoch}: train_acc={train_acc:.3f}, val_acc={val_acc:.3f}")


## Lab Questions

1. Add a causal mask to `scaled_dot_product_attention`. How do outputs change?
2. Compare `num_heads` in {1, 2, 4, 8}. Which setting gives the best validation accuracy?
3. Replace `LayerNorm` with no normalization. What happens to training stability?
4. Plot attention heatmaps for two different heads. Do they focus on the same positions?
5. Extend the classifier with a second attention block. Does performance improve?
